# 02 - Pair-level splitting for benchmark datasets (OP3, Tahoe, Novartis)

This is **stage 2** of a two-stage split.

- **Stage 1** (notebook 01): 75/25 outer split per benchmark; only the 75% outer-train portion was written to `raw_datasets/{op3, tahoe, novartis}/`. The 25% is the held-out benchmark test set kept as `.h5ad`.
- **Stage 2** (this notebook): pair-level LPM-style train / val / test on the 75% outer-train pool.

## Strategy

Mirrors `lpm_style/notebooks_work/02_LPM_style_splitting.ipynb`: the split unit is the **pair** `(context, perturbation)`, not the compound. A single CID can have one pair in train and another in val (across different cell lines / cell types); only the pairs are disjoint.

For each benchmark `(context, perturbation)` pair:

1. **Pair already in LPM** -> inherit LPM's split (frozen). This is what prevents Tahoe / Novartis from leaking, because their cell lines can already appear in LPM train.
2. **New benchmark pair** (not in LPM):
   - `perturbation in lpm_train_cids` -> **eligible** for val/test; enters the LPM-style greedy budgeted holdout selection (per-dataset 30% cap, ctx-train >= 1, pert-train >= 1, multi-dataset pairs first).
   - `perturbation not in lpm_train_cids` -> **forced to train** so the compound picks up an embedding this round.
3. Holdout pairs are partitioned 50/50 into val/test **by perturbation** so val perts and test perts are disjoint.

## Inputs

- `lpm_style/.plib_cache/annotations/df_annot_split.parquet` (9 LPM datasets, with pre-existing split labels; never overwritten).
- `lpm_style/.plib_cache/raw_datasets/{op3, tahoe, novartis}/*.parquet` (75% outer-train, from notebook 01).

## Outputs

- `lpm_style/.plib_cache/annotations/df_annot_split_extended.parquet` (LPM 9 datasets + 3 benchmarks, unified pair-level split).

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PLIBDATA_ROOT = '/home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets'
ANNOT_DIR     = Path('/home/icb/olga.novitskaia/lpm_style/.plib_cache/annotations')
LPM_ANNOT     = ANNOT_DIR / 'df_annot_split.parquet'
OUT_ANNOT     = ANNOT_DIR / 'df_annot_split_extended.parquet'

BENCHMARK_KEYS = ['op3', 'tahoe', 'novartis']

RATIO_TRAIN, RATIO_VAL, RATIO_TEST = 0.70, 0.15, 0.15
SPLIT_SEED = 42

## Helpers

In [3]:
def read_benchmark_annot(dataset_key: str) -> pd.DataFrame:
    """Read all parquet files of one benchmark dataset and return the unique
    pair-level annotation (dataset, context, perturbation, log_dose, time).
    """
    folder = f'{PLIBDATA_ROOT}/{dataset_key}'
    cols = ['dataset', 'context', 'perturbation', 'log_dose', 'time']
    dfs = []
    for f in sorted(os.listdir(folder)):
        if not f.endswith('.parquet'):
            continue
        df = pd.read_parquet(f'{folder}/{f}', columns=cols).drop_duplicates()
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True).drop_duplicates().reset_index(drop=True)
    out['perturbation'] = out['perturbation'].astype(str)
    out['context'] = out['context'].astype(str)
    return out

In [4]:
def greedy_pair_holdout(
    eligible_pairs,
    pair_to_datasets,
    ctx_train_count,
    pert_train_count,
    dataset_holdout_cap,
    dataset_holdout_used,
    seed: int = SPLIT_SEED,
) -> dict:
    """LPM-style greedy budgeted holdout selection at the pair level.

    Mirrors `lpm_style/notebooks_work/02_LPM_style_splitting.ipynb` (cell 14):
      - multi-dataset pairs first (hardest to fit; consume budget in every dataset they touch),
      - tie-break is seeded random,
      - a pair is accepted to holdout only if all three hold:
          * per-dataset cap not exceeded in ANY of the pair's datasets,
          * ctx_train_count[c] > 1  (removing keeps >=1 train pair for the context),
          * pert_train_count[p] > 1 (removing keeps >=1 train pair for the perturbation).

    Counts are mutated in place so each decision sees the current state.
    Returns {pair: 'holdout' or 'train'} for the eligible pairs only.
    """
    rng = np.random.RandomState(seed)
    pair_n_ds = {p: len(pair_to_datasets[p]) for p in eligible_pairs}
    pair_rand = {p: rng.random() for p in eligible_pairs}
    order = sorted(eligible_pairs, key=lambda p: (-pair_n_ds[p], pair_rand[p]))

    decision = {p: 'train' for p in eligible_pairs}
    n_accept = n_rej_budget = n_rej_ctx = n_rej_pert = 0
    for pair in order:
        c, pert = pair
        dss = pair_to_datasets[pair]
        if ctx_train_count[c] <= 1:
            n_rej_ctx += 1
            continue
        if pert_train_count[pert] <= 1:
            n_rej_pert += 1
            continue
        if any(dataset_holdout_used[d] + 1 > dataset_holdout_cap[d] for d in dss):
            n_rej_budget += 1
            continue
        decision[pair] = 'holdout'
        ctx_train_count[c] -= 1
        pert_train_count[pert] -= 1
        for d in dss:
            dataset_holdout_used[d] += 1
        n_accept += 1

    print(f'eligible pairs:                          {len(eligible_pairs):,}')
    print(f'  accepted to holdout:                   {n_accept:,}')
    print(f'  rejected - dataset budget exhausted:   {n_rej_budget:,}')
    print(f'  rejected - last train pair for ctx:    {n_rej_ctx:,}')
    print(f'  rejected - last train pair for pert:   {n_rej_pert:,}')
    return decision

## Load existing LPM annotation and build the train perturbagen set

In [5]:
df_lpm = pd.read_parquet(LPM_ANNOT)
df_lpm['perturbation'] = df_lpm['perturbation'].astype(str)
df_lpm['context'] = df_lpm['context'].astype(str)

print('existing LPM annotation:')
print(df_lpm['dataset'].value_counts())
print()
print(df_lpm['split'].value_counts())

existing LPM annotation:
dataset
LINCS_phase1_level3_epsilon            115511
CIGS MCE                                21819
LINCS_phase2_level3                     15591
CIGS TCM                                 3609
Ginkgo VCPI vcpi-0001 (tvc-bhr-009)      2272
Ginkgo VCPI vcpi-0002 (tvc-kdl-010)      1488
srivatsan20_sciplex3                      560
Ginkgo GDPx2                              344
dilimap_train                             250
Name: count, dtype: int64

split
train    114137
val       23661
test      23646
Name: count, dtype: int64


In [6]:
lpm_train_cids = set(df_lpm.loc[df_lpm['split'] == 'train', 'perturbation'])
print(f'unique CIDs in LPM train: {len(lpm_train_cids):,}')

unique CIDs in LPM train: 35,044


## Read benchmark per-pair annotations

In [7]:
benchmark_annot_per_ds = {}
for key in BENCHMARK_KEYS:
    annot = read_benchmark_annot(key)
    benchmark_annot_per_ds[key] = annot
    n_rows = len(annot)
    n_pairs = annot[['context', 'perturbation']].drop_duplicates().shape[0]
    n_cids = annot['perturbation'].nunique()
    n_ctx = annot['context'].nunique()
    print(f'[{key}] rows={n_rows:,}  unique_pairs={n_pairs:,}  unique_cids={n_cids:,}  unique_contexts={n_ctx:,}')

[op3] rows=415  unique_pairs=415  unique_cids=105  unique_contexts=4
[tahoe] rows=40,336  unique_pairs=13,612  unique_cids=284  unique_contexts=48
[novartis] rows=11,460  unique_pairs=2,865  unique_cids=2,865  unique_contexts=1


## Build unified pair table and classify each benchmark pair

Three classes of benchmark pair:

- **inherited** — `(context, perturbation)` already exists in LPM. The pair keeps LPM's split label. Most Tahoe / Novartis overlap with LPM cell lines falls here, which is what eliminates the pair-level leak.
- **eligible** — new pair (not in LPM) whose `perturbation` is in `lpm_train_cids`. Candidate for val/test in the greedy step below.
- **forced-train** — new pair whose `perturbation` is not in `lpm_train_cids`. The CID has no LPM embedding yet, so the pair goes to train so the embedding gets gradient this round.

In [8]:
assert df_lpm.groupby(['context', 'perturbation'])['split'].nunique().max() == 1, \
    'LPM annotation has a pair labelled with multiple splits'

lpm_pair_split = (
    df_lpm.drop_duplicates(['context', 'perturbation'])
    .set_index(['context', 'perturbation'])['split']
    .to_dict()
)

bench_pairs_per_ds = {}
for key in BENCHMARK_KEYS:
    bench_pairs_per_ds[key] = (
        benchmark_annot_per_ds[key][['context', 'perturbation']]
        .drop_duplicates()
        .reset_index(drop=True)
    )

bench_pairs_long = pd.concat(
    [bench_pairs_per_ds[k].assign(benchmark=k) for k in BENCHMARK_KEYS],
    ignore_index=True,
)
pair_to_benchmarks = (
    bench_pairs_long.groupby(['context', 'perturbation'])['benchmark']
    .agg(lambda s: tuple(sorted(set(s))))
    .to_dict()
)

bench_unique_pairs = list(pair_to_benchmarks.keys())
inherited_pairs    = [p for p in bench_unique_pairs if p in lpm_pair_split]
bench_only_pairs   = [p for p in bench_unique_pairs if p not in lpm_pair_split]
eligible_pairs     = [p for p in bench_only_pairs if p[1] in lpm_train_cids]
forced_train_pairs = [p for p in bench_only_pairs if p[1] not in lpm_train_cids]

print(f'benchmark unique pairs:    {len(bench_unique_pairs):,}')
print(f'  inherited from LPM:      {len(inherited_pairs):,}')
print(f'  benchmark-only pairs:    {len(bench_only_pairs):,}')
print(f'    eligible (pert in LPM train CIDs):    {len(eligible_pairs):,}')
print(f'    forced-train (pert not in LPM train): {len(forced_train_pairs):,}')

benchmark unique pairs:    16,892
  inherited from LPM:      236
  benchmark-only pairs:    16,656
    eligible (pert in LPM train CIDs):    13,251
    forced-train (pert not in LPM train): 3,405


In [9]:
pair_split = dict(lpm_pair_split)
for p in bench_unique_pairs:
    pair_split.setdefault(p, 'train')

ctx_train_count = defaultdict(int)
pert_train_count = defaultdict(int)
for (c, p), s in pair_split.items():
    if s == 'train':
        ctx_train_count[c] += 1
        pert_train_count[p] += 1

bench_total_pairs = {k: bench_pairs_per_ds[k].shape[0] for k in BENCHMARK_KEYS}
HOLDOUT_PORTION = RATIO_VAL + RATIO_TEST
bench_holdout_cap = {k: int(bench_total_pairs[k] * HOLDOUT_PORTION) for k in BENCHMARK_KEYS}
bench_holdout_used = {k: 0 for k in BENCHMARK_KEYS}
for pair in inherited_pairs:
    if pair_split[pair] in ('val', 'test'):
        for d in pair_to_benchmarks[pair]:
            bench_holdout_used[d] += 1

print('per-benchmark pair totals:    ', bench_total_pairs)
print('per-benchmark holdout cap 30%:', bench_holdout_cap)
print('holdout used by inherited LPM:', bench_holdout_used)
print()

decision = greedy_pair_holdout(
    eligible_pairs,
    pair_to_benchmarks,
    ctx_train_count,
    pert_train_count,
    bench_holdout_cap,
    bench_holdout_used,
)
for pair, d in decision.items():
    if d == 'holdout':
        pair_split[pair] = 'holdout'

rng_vt = np.random.RandomState(SPLIT_SEED + 5000)
holdout_perts = sorted({pert for (c, pert), s in pair_split.items() if s == 'holdout'})
rng_vt.shuffle(holdout_perts)
n_val_perts = int(round(len(holdout_perts) * RATIO_VAL / HOLDOUT_PORTION))
val_perts = set(holdout_perts[:n_val_perts])
test_perts = set(holdout_perts[n_val_perts:])

for pair, s in list(pair_split.items()):
    if s == 'holdout':
        pair_split[pair] = 'val' if pair[1] in val_perts else 'test'

assert val_perts.isdisjoint(test_perts), 'val and test perturbations overlap'
print()
print(f'holdout perturbations: {len(holdout_perts):,} -> val={len(val_perts):,}, test={len(test_perts):,}')
print(f'unified pair split counts: {dict(Counter(pair_split.values()))}')

per-benchmark pair totals:     {'op3': 415, 'tahoe': 13612, 'novartis': 2865}
per-benchmark holdout cap 30%: {'op3': 124, 'tahoe': 4083, 'novartis': 859}
holdout used by inherited LPM: {'op3': 0, 'tahoe': 102, 'novartis': 0}

eligible pairs:                          13,251
  accepted to holdout:                   4,964
  rejected - dataset budget exhausted:   8,287
  rejected - last train pair for ctx:    0
  rejected - last train pair for pert:   0

holdout perturbations: 1,115 -> val=558, test=557
unified pair split counts: {'train': 124537, 'val': 23314, 'test': 22933}


## Broadcast pair splits to rows and merge with LPM

In [10]:
df_bench_all = []
for key in BENCHMARK_KEYS:
    annot = benchmark_annot_per_ds[key].copy()
    pair_keys = list(zip(annot['context'], annot['perturbation']))
    annot['split'] = [pair_split.get(k) for k in pair_keys]
    n_unassigned = annot['split'].isna().sum()
    if n_unassigned:
        print(f'WARNING [{key}]: {n_unassigned} rows with no pair assignment (dropped)')
        annot = annot.dropna(subset=['split'])
    df_bench_all.append(annot)
df_bench = pd.concat(df_bench_all, ignore_index=True)
print(df_bench.groupby(['dataset', 'split']).size().unstack(fill_value=0))

split                     test  train   val
dataset                                    
Novartis MoABox DRUG-seq  1788   8024  1648
op3                         66    291    58
tahoe100                  5672  28220  6444


In [11]:
df_annot_split_extended = pd.concat(
    [df_lpm[['dataset', 'context', 'perturbation', 'log_dose', 'time', 'split']],
     df_bench[['dataset', 'context', 'perturbation', 'log_dose', 'time', 'split']]],
    ignore_index=True,
)
print(df_annot_split_extended.groupby(['dataset', 'split']).size().unstack(fill_value=0))

split                                 test  train    val
dataset                                                 
CIGS MCE                              3281  15274   3264
CIGS TCM                               559   2527    523
Ginkgo GDPx2                            54    241     49
Ginkgo VCPI vcpi-0001 (tvc-bhr-009)      0   2269      3
Ginkgo VCPI vcpi-0002 (tvc-kdl-010)      1   1487      0
LINCS_phase1_level3_epsilon          17243  80858  17410
LINCS_phase2_level3                   2383  10914   2294
Novartis MoABox DRUG-seq              1788   8024   1648
dilimap_train                           35    175     40
op3                                     66    291     58
srivatsan20_sciplex3                    90    392     78
tahoe100                              5672  28220   6444


## Sanity check 1 — pair-level disjointness

After adding OP3 / Tahoe / Novartis on top of LPM, no `(context, perturbation)` pair should appear in more than one split.

In [12]:
n_splits_per_pair = (
    df_annot_split_extended
    .groupby(['context', 'perturbation'])['split']
    .nunique()
)
violators = n_splits_per_pair[n_splits_per_pair > 1]
print(f'pairs spanning multiple splits: {len(violators)}')
if len(violators):
    print(violators.head(20))
assert violators.empty, f'{len(violators)} (context, perturbation) pairs span multiple splits'

pairs spanning multiple splits: 0


## Sanity check 2 — learnability

Every perturbation and every context that appears in val/test should also appear in train somewhere (in the unified pool), so its embedding receives gradient. The greedy selection's `ctx_train_count >= 1` and `pert_train_count >= 1` rules + the forced-train branch for non-LPM CIDs together guarantee this; this check just confirms it.

In [14]:
pert_train = set(df_annot_split_extended.loc[df_annot_split_extended['split'] == 'train', 'perturbation'])
ctx_train  = set(df_annot_split_extended.loc[df_annot_split_extended['split'] == 'train', 'context'])
pert_holdo = set(df_annot_split_extended.loc[df_annot_split_extended['split'].isin(['val', 'test']), 'perturbation'])
ctx_holdo  = set(df_annot_split_extended.loc[df_annot_split_extended['split'].isin(['val', 'test']), 'context'])

pert_val  = set(df_annot_split_extended.loc[df_annot_split_extended['split'] == 'val',  'perturbation'])
pert_test = set(df_annot_split_extended.loc[df_annot_split_extended['split'] == 'test', 'perturbation'])

pair_col = list(zip(df_annot_split_extended['context'], df_annot_split_extended['perturbation']))
df_annot_split_extended_pairs = pd.Series(pair_col, index=df_annot_split_extended.index)
pairs_train = set(df_annot_split_extended_pairs[df_annot_split_extended['split'] == 'train'])
pairs_val   = set(df_annot_split_extended_pairs[df_annot_split_extended['split'] == 'val'])
pairs_test  = set(df_annot_split_extended_pairs[df_annot_split_extended['split'] == 'test'])

print(f'perturbations only in val/test (no train pair anywhere): {len(pert_holdo - pert_train)}')
print(f'contexts      only in val/test (no train pair anywhere): {len(ctx_holdo - ctx_train)}')
print(f'perturbations shared between val and test:               {len(pert_val & pert_test)}')
print(f'pairs in BOTH train and val :                            {len(pairs_train & pairs_val)}')
print(f'pairs in BOTH train and test:                            {len(pairs_train & pairs_test)}')
print(f'pairs in BOTH val   and test:                            {len(pairs_val   & pairs_test)}')

perturbations only in val/test (no train pair anywhere): 0
contexts      only in val/test (no train pair anywhere): 0
perturbations shared between val and test:               390
pairs in BOTH train and val :                            0
pairs in BOTH train and test:                            0
pairs in BOTH val   and test:                            0


## Sanity check 3 — per-dataset split summary

In [15]:
report = (
    df_annot_split_extended.groupby(['dataset', 'split']).size()
    .unstack(fill_value=0)
    .reindex(columns=['train', 'val', 'test'], fill_value=0)
)
report['total'] = report.sum(axis=1)
for c in ['train', 'val', 'test']:
    report[f'{c}_frac'] = (report[c] / report['total']).round(3)
report

split,train,val,test,total,train_frac,val_frac,test_frac
dataset,,,,,,,
CIGS MCE,15274,3264,3281,21819,0.700,0.150,0.150
CIGS TCM,2527,523,559,3609,0.700,0.145,0.155
Ginkgo GDPx2,241,49,54,344,0.701,0.142,0.157
Ginkgo VCPI vcpi-0001 (tvc-bhr-009),2269,3,0,2272,0.999,0.001,0.000
Ginkgo VCPI vcpi-0002 (tvc-kdl-010),1487,0,1,1488,0.999,0.000,0.001
LINCS_phase1_level3_epsilon,80858,17410,17243,115511,0.700,0.151,0.149
LINCS_phase2_level3,10914,2294,2383,15591,0.700,0.147,0.153
Novartis MoABox DRUG-seq,8024,1648,1788,11460,0.700,0.144,0.156
dilimap_train,175,40,35,250,0.700,0.160,0.140


## Save the extended annotation

In [16]:
df_annot_split_extended.to_parquet(OUT_ANNOT, index=False)
print(f'wrote {OUT_ANNOT}')

wrote /home/icb/olga.novitskaia/lpm_style/.plib_cache/annotations/df_annot_split_extended.parquet


In [18]:
df_annot_split_extended

,dataset,context,perturbation,log_dose,time,split
0,CIGS MCE,CVCL_0063,5281081,1.0,24.0,train
1,CIGS MCE,CVCL_0063,11957668,1.0,24.0,val
2,CIGS MCE,CVCL_0063,127264445,1.0,24.0,train
3,CIGS MCE,CVCL_0063,60651,1.0,24.0,train
4,CIGS MCE,CVCL_0063,6034,1.0,24.0,train
...,...,...,...,...,...,...
213650,Novartis MoABox DRUG-seq,CVCL_0042,11467730,-1.0,24.0,val
213651,Novartis MoABox DRUG-seq,CVCL_0042,9821217,-2.0,24.0,train
213652,Novartis MoABox DRUG-seq,CVCL_0042,9821217,-1.0,24.0,train
213653,Novartis MoABox DRUG-seq,CVCL_0042,9821217,0.0,24.0,train
